In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import librosa
from sklearn.pipeline import make_pipeline
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from math import log10
import parselmouth
from maad import sound as suono
from maad import features as ft
from maad.util import power2dB
from spafe.features.lfcc import lfcc

In [2]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [3]:
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [4]:
cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), 
                csvTotale.drop(columns=['path', 'age']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean()

# BASELINE = -7.24

np.float64(-7.242994400990241)

In [5]:
audioDevelopment = {file: librosa.load('.././data/audios_development/'+file, sr=22050)[0] for file in csvTotale['path']}

In [6]:
def getMFCC(audio, imp=0):  # Keep imp=1 otherwise the score worsens
    numFcc = 35
    mfccs = librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=numFcc)
    if imp:
        energy = np.sum(librosa.amplitude_to_db(np.abs(librosa.stft(audio))**2, ref=np.max))
        
        deltas = librosa.feature.delta(mfccs).sum(axis=1)
        double_deltas = librosa.feature.delta(mfccs, order=2).sum(axis=1)

        return pd.Series(np.concatenate((mfccs.mean(axis=1), deltas, double_deltas, [energy]), axis=0),
                            index=[f'mfcc_{i}' for i in range(numFcc)]+['delta_'+str(i) for i in range(numFcc)]+['double_delta_'+str(i) for i in range(numFcc)]+['energy'])
    else:
        return pd.Series(mfccs.mean(axis=1), index=[f'mfcc_mean{i}' for i in range(numFcc)])

In [7]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 150) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [8]:
def computeF0(audio):
    sound = parselmouth.Sound(audio)
    pitch = sound.to_pitch()
    info = str(pitch.info()).split('\n')
    return pd.Series([pitch.count_voiced_frames(), pitch.get_mean_absolute_slope(), pitch.xmax-pitch.xmin, 
                      pitch.n_frames, *[float(info[15+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 5)],
                      *[float(info[21+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 3)]],
                     index=['nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 
                            'q10', 'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10']
                     )

In [9]:
def computeMath(audio):
    return pd.Series([skew(audio), kurtosis(audio), np.mean(np.abs(hilbert(audio))), np.mean(np.abs(np.fft.fft(audio)))],
                     index=['skew', 'kurtosis', 'hilbertMean', 'fftMean'])

In [10]:
def computSNR(audio):
    return pd.Series([suono.temporal_snr(audio)[-1]], index=['temporalSNR'])

In [11]:
def computeTemporalMedia(audio):
    return pd.Series([ft.temporal_median(audio)], index=['temporalMedian'])

In [12]:
def computeAllFeatures(audio):
    return pd.Series(ft.all_temporal_features(audio, fs=22050, nperseg=256).values[0],
                     index=['sm', 'sv', 'ss', 'sk', 'Time 5%', "Time 25%", "Time 50%", "Time 75%", "Time 95%  ", 
                            "zcr", "duration_5", "duration_90"])

In [13]:
def computePeakFrequency(audio):
    return pd.Series(ft.peak_frequency(audio, fs=22050, nperseg=256, amp=True),
                     index=['peakFrequency', 'peakFrequencyAmp'])

In [14]:
def computeEntropy(audio):
    return pd.Series(ft.temporal_entropy(audio), index=['entropy'])

In [15]:
def computeSpectralActivity(audio):
    return pd.Series(ft.spectral_activity(power2dB(suono.median_equalizer(suono.spectrogram(audio, 22050)[0])), dB_threshold=50)[-1],
                     index=['spectralActivity'])

In [16]:
def computeSpectralCover(audio):
    Sxx, tn, fn, ext = suono.spectrogram(audio, 22050)
    return pd.Series(ft.spectral_cover(power2dB(suono.median_equalizer(Sxx)), fn=fn, 
                                       flim_LF=(0, 120), flim_MF=(120, 180), flim_HF=(180, 300)),
                     index=['LFC', 'MFC', 'HFC']) 

In [17]:
# NO BUENO JUST NOISE
def computeMaadStats(audio):
    Sxx_power, tn, fn, _ = suono.spectrogram(audio, 22050)      
    return pd.Series([ft.number_of_peaks(Sxx_power, fn=fn, nperseg=256), 
                      ft.bioacoustics_index(Sxx_power, fn=fn, flim=(100, 3000)),
                      ft.acoustic_diversity_index(Sxx_power, fn=fn, fmin=80, fmax=3000),
                      ft.acoustic_eveness_index(Sxx_power, fn=fn, fmin=80, fmax=3000),
                      ],
                    index=['nPeaks', 'bioIndex', 'acousticDiversity', 'acousticEvenness'])

In [18]:
# NO BUENO JUST NOISE
def getSpectralContrast(audio):
    return pd.Series(librosa.feature.spectral_contrast(S=np.abs(librosa.stft(audio)), sr=22050, fmin=80, n_fft=256, hop_length=32).sum(axis=1),
                     index=[f'spectralContrast{i}' for i in range(7)])

In [19]:
def getLogMel(audio):
    return pd.Series(librosa.amplitude_to_db(librosa.feature.melspectrogram(y=audio, sr=22050, n_fft=256, hop_length=32, n_mels=20).mean(axis=1)),
                     index=[f'mel{i}' for i in range(20)])

In [20]:
def computeRMS(audio):
    rms = librosa.feature.rms(y=audio)
    return pd.Series([20*log10(rms.mean()), rms.std(), rms.max(), 20*log10(rms.sum())], index=['rms_mean', 'rms_std', 'rms_max', 'rms_sum'])

In [21]:
def getLFCC(audio):
    return pd.Series(lfcc(audio, fs=22050, num_ceps=15, low_freq=50, high_freq=3000, nfft=256).mean(axis=0),
                        index=[f'lfcc{i}' for i in range(15)])

In [22]:
def computeMetrics(audios):
    return pd.DataFrame({file: pd.concat([
            getMFCC(audios[file]),
            getSpectralEnergy(audios[file]),
            computeF0(audios[file]),
            computeMath(audios[file]),
            computSNR(audios[file]),
            computeTemporalMedia(audios[file]),
            computeAllFeatures(audios[file]),
            computePeakFrequency(audios[file]),
            computeEntropy(audios[file]),   # fucking entropy drops the score
            # computeSpectralActivity(audios[file]),
            # computeSpectralCover(audios[file]),
            # getLogMel(audios[file]),
            # getLFCC(audios[file]),
            ], axis=0)
        for file in audios}).T

In [23]:
metrics = pd.concat([computeMetrics(audioDevelopment), 
                    csvTotale.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)

metrics['log_hnr']= metrics['hnr'].map(lambda x: 20*log10(np.abs(x)))

In [31]:
cross_val_score(HistGradientBoostingRegressor(), metrics,
                 csvTotale['age'], cv=20, scoring='neg_mean_absolute_error').mean()

np.float64(-6.681333772375254)

In [25]:
for item, imp in sorted(zip(list(metrics.columns)+['age'], 
                            pd.concat([metrics, csvTotale.set_index(csvTotale['path'])['age']], axis=1).corr('spearman')['age']), key=lambda x: abs(x[1]), reverse=True)[1:]:
    print(f'{item}: {imp}')

nFrames: 0.6022603562876916
duration: 0.602252908183809
Time 95%  : 0.6016573045219915
duration_90: 0.5994765605409939
duration_5: 0.5948130345006974
Time 75%: 0.5904688371671031
Time 50%: 0.5818011133930301
nVoicedFrames: 0.5644983266317832
Time 25%: 0.5518922891861292
Time 5%: 0.5350617896196377
hnr: -0.5326292343962696
log_hnr: 0.5320784492500874
spectralEnergy250-650: 0.5186342802478199
sm: 0.5010898324353773
fftMean: 0.49037696789958823
spectralEnergy1000-8000: 0.4742084761261652
max_pitch: 0.4694639426920942
min_pitch: -0.46452284591811516
temporalSNR: 0.4389244252671011
mfcc_mean6: -0.4222780140021344
ss: -0.41707174962926924
skew: -0.4170666241622713
mean_pitch: 0.36619218880953386
zcr: 0.3610859442341537
jitter: 0.34920179676737306
mfcc_mean13: -0.34195572156396453
mfcc_mean33: -0.32793932338185033
kurtosis: 0.32287861765139875
sk: 0.3228641469774155
mfcc_mean15: -0.3187933132261265
mfcc_mean17: -0.31683729202774413
mfcc_mean31: -0.2983493012563597
mfcc_mean2: -0.2800573307206

In [26]:
audioEval = {file: librosa.load('.././data/audios_evaluation/'+file, sr=22050)[0] for file in csvEval['path']}

In [27]:
metricsEval = pd.concat([computeMetrics(audioEval), 
                    csvEval.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)
metricsEval['log_hnr']= metricsEval['hnr'].map(lambda x: 20*log10(np.abs(x)))

In [28]:
ypred = pd.Series(HistGradientBoostingRegressor().fit(metrics, csvTotale['age'])
          .predict(metricsEval),
          name='Predicted', index=csvEval.index)

In [ ]:
from sklearn.metrics import root_mean_squared_error
pd.DataFrame({"nowModel":[
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9322.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9555.csv', header=0, index_col=0)['Predicted'], ypred),
    root_mean_squared_error(pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted'], ypred)
    ],
 'bestModel':[0,
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9308.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9650.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predicted1501T1827-9100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('submissionOggi.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('predictedNoHIST-9426.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'], 
                            pd.read_csv('predicted-9322.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9235.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9555.csv', header=0, index_col=0)['Predicted']),
    root_mean_squared_error(pd.read_csv('9-100.csv', header=0, index_col=0)['Predicted'],
                            pd.read_csv('predicted-9196.csv', header=0, index_col=0)['Predicted'])
     ],
 })

,nowModel,bestModel
0,2.578712,0.000000
1,3.455323,2.344467
2,2.648260,3.246054
3,3.030260,3.056776
4,3.198745,3.268457
5,3.258811,3.378994
6,3.244530,3.284294
7,4.729119,4.904805
8,3.075733,3.094170


In [30]:
ypred.to_csv('predicted-bho.csv', header=True)